# Train the tobacco-leaf classifier on a Colab GPU
Run the cells **top to bottom**. Before you start: *Runtime → Change runtime type → T4 GPU*.
The dataset is downloaded from Hugging Face automatically. Nothing private is needed, so **do not type any keys or passwords** into this notebook.

In [ ]:
# 1. Is a GPU visible?
import torch
print("GPU available:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

**2. Upload `ml_for_colab.zip`** (made on your PC with `python colab/make_zip.py` inside the `ml` folder). It holds code only.

In [ ]:
%cd /content
!rm -f /content/ml_for_colab*.zip
from google.colab import files
up = files.upload()            # choose ml_for_colab.zip (NOT make_zip.py)
zips = [n for n in up if n.endswith(".zip")]
assert zips, "Please upload the zip file made by make_zip.py (ml_for_colab.zip)"
!rm -rf /content/ml && mkdir /content/ml
!unzip -q -o "/content/{zips[0]}" -d /content/ml
%cd /content/ml
!pip -q install datasets scikit-learn pyyaml pillow
%env HF_HUB_DISABLE_PROGRESS_BARS=1

## 3. Tuning trials (validation only)
Each trial adds one idea to the previous one, so you can see what helped. They use `--skip-test`, so **the test set is not used to choose anything**.
I have not timed these on a GPU, so the first trial doubles as a speed check.

In [ ]:
COMMON = "--epochs 30 --lr 0.005 --cosine --skip-test"
!python -m src.training.centralised $COMMON --out r1_cosine

In [ ]:
!python -m src.training.centralised $COMMON --strong-aug --label-smoothing 0.1 --out r2_aug

In [ ]:
!python -m src.training.centralised $COMMON --strong-aug --label-smoothing 0.1 --balance --out r3_balance

In [ ]:
!python -m src.training.centralised $COMMON --strong-aug --label-smoothing 0.1 --balance --img-size 224 --out r4_224px

## 4. Compare on VALIDATION macro-F1 and pick one

In [ ]:
import json, pathlib
rows = []
for d in sorted(pathlib.Path("outputs").glob("r*")):
    r = json.loads((d / "result.json").read_text())
    h = r["history"]
    last5 = sum(e["valMacroF1"] for e in h[-5:]) / 5
    rows.append((d.name, r["bestValMacroF1"], last5, max(e["valAccuracy"] for e in h)))
print(f"{'run':14} {'best val F1':>12} {'mean of last 5':>15} {'best val acc':>13}")
for n, b, l, a in sorted(rows, key=lambda t: -t[1]):
    print(f"{n:14} {b:12.3f} {l:15.3f} {a:13.3f}")
print("Note: the last-5 average shows how stable training is; a high best score alone can be luck.")

## 5. Test the chosen run ONCE
Put the run's folder name below. The script refuses to run a second time on the same folder, so the test set stays honest.

In [ ]:
CHOSEN = "r4_224px"   # <- change to the run you picked above
!python -m src.training.final_test --out $CHOSEN

In [ ]:
# 6. Download the chosen model. Save it on your PC as ml/outputs/<CHOSEN>/model.pt and result.json
import shutil
shutil.make_archive("chosen_model", "zip", f"outputs/{CHOSEN}")
files.download("chosen_model.zip")

## Part 2: use more data (do this after Part 1)
Two ways to use more photos. **Step A** is an experiment judged on validation only. **Step B** makes the final model.

**Step A: add the dataset's synthetic training images** (about 173 extra copies of rare diseases, training only, never used for validation or test). This run uses the same recipe as r4, so the two can be compared fairly.

In [ ]:
COMMON = "--epochs 30 --lr 0.005 --cosine --label-smoothing 0.1 --strong-aug --balance --img-size 224 --skip-test"
!python -m src.training.centralised $COMMON --include-synthetic --out r5_synthetic

In [ ]:
import json, pathlib
for name in ["r4_224px", "r5_synthetic"]:
    f = pathlib.Path("outputs") / name / "result.json"
    if not f.exists():
        print(name, "not found in this session (re-run that trial, or compare with the numbers you noted)"); continue
    h = json.loads(f.read_text())["history"]
    last5 = sum(e["valMacroF1"] for e in h[-5:]) / 5
    print(f"{name:14} best val F1 {max(e['valMacroF1'] for e in h):.3f}   mean of last 5 {last5:.3f}")
print("Earlier r4_224px numbers: best 0.840, mean of last 5 0.795")

**Decide:** use the synthetic images in the final model only if the *mean of the last 5* clearly beats r4 (r4 = 0.795). A gap of 0.01 or 0.02 is noise on 138 validation photos.

**Step B: the final model on ALL photos** (train + validation + test, optionally + synthetic). After this there are no unseen photos left, so **no new score can be measured**. Keep quoting the earlier held-out score (84.1% accuracy, macro-F1 0.729) as the estimate. It also keeps the last epoch (a fixed 30 epochs) instead of the best-validation epoch.

In [ ]:
FINAL_FLAGS = ""   # set to "--include-synthetic" if Step A showed a clear gain
!python -m src.training.centralised --epochs 30 --lr 0.005 --cosine --label-smoothing 0.1 --strong-aug --balance --img-size 224 $FINAL_FLAGS --train-on-all --out final_all

In [ ]:
import shutil
shutil.make_archive("final_all", "zip", "outputs/final_all")
files.download("final_all.zip")